# Code-Mixed Hinglish and Marathi Social Text Analytics with LLM-Assisted Annotation
## Big Data and Social Media Analytics — Master Experimental Pipeline Walkthrough

This notebook demonstrates the end-to-end distributed Big Data and Indic NLP pipeline:
1. **Big Data Ingestion & PySpark Processing**: Ingesting multi-source code-mixed social datasets (SentiMix, MahaSent, HASOC, GLUECoS).
2. **Multi-Annotator Human Gold Benchmarking**: Inter-rater reliability (Cohen's Kappa & Krippendorff's Alpha).
3. **LLM Annotation Engine**: Zero-Shot, Few-Shot, and Reasoning-Aware (Chain-of-Thought) prompting.
4. **LLM vs. Human Gold Evaluation**: Precision, Recall, Macro-F1, Kappa, and Subgroup Breakdown.
5. **Downstream Indic NLP Models**: TF-IDF Baselines vs. MuRIL, IndicBERT, XLM-RoBERTa across 3 Regimes (Gold-only, LLM-only, Mixed).
6. **Confidence Calibration**: Expected Calibration Error (ECE) and Reliability Diagrams.
7. **Cost vs. Quality Trade-Off**: Hybrid budget Pareto optimization.
8. **Error Analysis**: Pragmatic failure modes (Sarcasm, Polysemy, Mixed Scripts).

In [ ]:
import os
import sys
import pandas as pd
import numpy as np

# Ensure project root is available
sys.path.insert(0, os.path.abspath('..'))

### Step 1: Distributed PySpark Preprocessing & Harmonization

In [ ]:
from bigdata.spark_preprocessing import preprocess_all_datasets

unified_df = preprocess_all_datasets()
print(f"Unified corpus created with {len(unified_df)} records.")
unified_df.head(5)

### Step 2: Human Gold Annotation & Inter-Rater Reliability

In [ ]:
from annotation.agreement import generate_agreement_report

agreement_df = generate_agreement_report(gold_dir="../data/gold")
agreement_df

### Step 3: LLM Synthetic Annotation across Paradigms

In [ ]:
from llm.annotator import annotate_dataset

for strategy in ["zero_shot", "few_shot", "reasoning_aware"]:
    annotate_dataset(strategy=strategy, input_path="../data/gold/gold_master.csv")
    print(f"Generated {strategy} annotations.")

### Step 4: LLM vs Human Gold Evaluation

In [ ]:
from llm.evaluator import evaluate_llm_vs_gold

eval_res = evaluate_llm_vs_gold(gold_path="../data/gold/gold_master.csv", llm_dir="../data/llm_annotations")
print("=== Performance across Prompting Paradigms ===")
print(eval_res["metrics_df"].to_string(index=False))

print("\n=== Subgroup Breakdown ===")
print(eval_res["subgroups_df"].head(10).to_string(index=False))

### Step 5: Downstream Indic NLP Model Benchmarking

In [ ]:
from models.evaluate import run_all_model_experiments

model_results_df = run_all_model_experiments(
    gold_path="../data/gold/gold_master.csv",
    llm_path="../data/llm_annotations/reasoning_aware.csv",
    output_path="../results/model_results.csv"
)
model_results_df

### Step 6: Cost vs Quality Hybrid Trade-Off

In [ ]:
from analytics.cost_quality import CostQualityExperiment

gold_df = pd.read_csv("../data/gold/gold_master.csv")
llm_df = pd.read_csv("../data/llm_annotations/reasoning_aware.csv")
cost_exp = CostQualityExperiment(gold_df, llm_df)
cost_df = cost_exp.run_experiment()
cost_df

### Step 7: Render All 9 Research Publication Figures

In [ ]:
from visualization.plots import generate_all_figures

generate_all_figures()